# Módulo 8 — Transformada de Fourier
## Seção 8.1 — Fundamentos de sinais, ondas e periodicidade

> **Estado editorial:** material didático fundamentado nas páginas indicadas; exercícios e exemplos numéricos são desenvolvimentos próprios. Python básico é suficiente. Dados sintéticos.

**Referência principal consultada:** Morettin, P. A.; Toloi, C. M. C. *Análise de séries temporais: modelos lineares univariados*, vol. 1, 3ª ed. (2018), cap. 13, §13.1, p. 335 e §13.2, pp. 335–336 (equações 13.1–13.5); apoio: §2.7.1, pp. 41–43 (harmônicas em modelos determinísticos) e §3.3, pp. 68–74 (sazonalidade). Nas seções seguintes do módulo serão estudados números complexos, séries de Fourier, DFT e FFT.

**Limite bibliográfico:** o capítulo 13 fundamenta o modelo harmônico; os desenvolvimentos pedagógicos sobre sinais contínuos, frequência de amostragem e exemplos simulados são nossos, não transcrições do livro. O mapeamento de videoaulas do projeto classifica Fourier como *mencionado*, não desenvolvido; portanto, não usamos as videoaulas como demonstração matemática.

### Objetivos e pré-requisitos

Ao final, distinguir sinal determinístico, série temporal observada e processo estocástico; definir periodicidade contínua e discreta; reconhecer senoides e sobreposição de harmônicas; calcular amostras manualmente; simular ruído e interpretar gráficos. Pré-requisitos: funções, seno e cosseno, esperança matemática (Módulo 1) e sazonalidade (Módulo 2). Cointegração (Módulo 7) **não** é necessária.

### 1. Intuição: o que a análise de Fourier procura resolver?

Uma série temporal pode oscilar porque vários mecanismos periódicos agem simultaneamente: ciclos diários, semanais, anuais. Um gráfico no tempo mostra **quando** a série sobe ou desce; uma descrição harmônica procura identificar **quais oscilações** contribuem para o comportamento. O livro, na p. 335, distingue duas situações de modelagem: frequências conhecidas, nas quais se estimam amplitudes e fases, e frequências desconhecidas, nas quais também é preciso estimar a frequência.

Cuidado: uma oscilação visual não prova a existência de uma periodicidade rigorosa, nem implica estacionariedade; ruído pode criar padrões aparentes.

### 2. Três objetos matemáticos diferentes

**Sinal determinístico:** função $s:\mathbb R\to\mathbb R$, com $s(t)$ fixo para cada $t$; por exemplo, $s(t)=2\cos(\pi t/2)$. **Processo estocástico:** família de variáveis aleatórias $\{Z_t:t\in T\}$; em geral, $Z_t=s(t)+\varepsilon_t$ é aleatório. **Série observada:** realização finita $(z_0,\ldots,z_{N-1})$. Não confundir trajetória realizada com a lei probabilística do processo.

### 3. Periodicidade no tempo contínuo

**Definição:** $s$ é periódica se existe $T>0$ tal que $s(t+T)=s(t)$ para todo $t$. O menor período positivo, quando existe, é o *período fundamental*. Para $R\ne0$ e $\omega\ne0$:

$$s(t)=R\cos(\omega t+\phi),\qquad T=\frac{2\pi}{|\omega|}.$$

**Demonstração:** $s(t+2\pi/|\omega|)=R\cos(\omega t+\phi+2\pi\operatorname{sgn}\omega)=s(t)$, pela periodicidade de $\cos$. A frequência angular $\omega$ mede **radianos por unidade de tempo**. Para sinal constante ($R=0$ ou $\omega=0$), não há período fundamental único.

### 4. Periodicidade em tempo discreto: condição importante

Uma sequência $s_n$ é periódica se existe inteiro $p\ge1$ tal que $s_{n+p}=s_n$ para todo inteiro $n$. Para $s_n=\cos(\omega n+\phi)$ com amplitude não nula e frequência não degenerada, a periodicidade discreta exige $\omega/(2\pi)$ racional (desconsiderando equivalências de frequência e casos degenerados). Se $\omega/(2\pi)=a/b$ na forma irredutível, $b$ é um período; o período fundamental pode ser menor em casos degenerados.

**Exemplo:** $s_n=\cos(\pi n/2)$ tem valores $1,0,-1,0,1,\ldots$, período 4. Em contrapartida, amostrar $s(t)=\cos(t)$ nos inteiros produz uma sequência que não possui período inteiro exato, pois $1/(2\pi)$ é irracional.

### 5. Somando ondas: construção de uma série

Um modelo determinístico com duas componentes é

$$s_t=\mu+R_1\cos(\omega_1t+\phi_1)+R_2\cos(\omega_2t+\phi_2).$$

Quando acrescentamos ruído centrado $\varepsilon_t$, obtemos $Z_t=s_t+\varepsilon_t$, semelhante ao modelo (13.1) de Morettin e Toloi, ampliado para dois harmônicos. **Esperança condicional ao mecanismo determinístico:** se $E(\varepsilon_t)=0$, então $E(Z_t)=s_t$. Não decorre daí que $Z_t$ seja fracamente estacionário, pois sua média varia com $t$.

**Exemplo aritmético:** $s_t=2+3\cos(\pi t/2)+\cos(\pi t)$. Nos instantes $t=0,1,2,3$ os valores são $6,1,0,1$; no instante 4, retorna a 6. Período fundamental 4. Confira substituindo cada valor na expressão, sem calculadora.

### 6. Laboratório 1 — construir sinais com NumPy

`numpy.arange(0,48)` cria os instantes; `np.cos` calcula a função elemento a elemento; `np.random.default_rng(2026)` torna o experimento reproduzível. `matplotlib.pyplot` permite comparar a curva limpa, as componentes e o sinal com ruído. A semente não torna uma realização prova matemática.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.arange(48)
mu = 2.0
comp1 = 3*np.cos(2*np.pi*t/12)
comp2 = 1.0*np.cos(2*np.pi*t/4)
sinal = mu + comp1 + comp2
rng = np.random.default_rng(2026)
observado = sinal + rng.normal(0, 0.65, len(t))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, observado, '.', alpha=.65, label='Observações com ruído')
ax.plot(t, sinal, '-', linewidth=2, label='Sinal determinístico')
ax.plot(t, mu+comp1, '--', label='Nível + componente de período 12')
ax.set(xlabel='Instante t', ylabel='Valor', title='Figura 8.1.1 — Sobreposição de harmônicas')
ax.legend()
plt.show()



**Interpretação:** a série resultante não tem necessariamente um único pico por período; a componente de período 4 modifica a forma da oscilação principal de período 12. Como 4 divide 12, o sinal somado mantém período 12. Ruído pode deslocar visualmente picos sem alterar o mecanismo determinístico.

### 7. Laboratório 2 — periodicidade discreta



In [ ]:
n = np.arange(16)
a = np.cos(np.pi*n/2)
print(np.round(a[:8], 5))
assert np.allclose(a[:-4], a[4:])
fig, ax = plt.subplots(figsize=(9,3))
ax.stem(n, a)
ax.set(xlabel='n', ylabel='s[n]', title='Figura 8.1.2 — Senoide discreta de período 4')
plt.show()



**Interpretação:** o `assert` verifica a identidade numericamente na amostra escolhida, mas não substitui a demonstração para todo $n$.

### 8. Limitações e erros comuns

- A existência de períodos individuais não garante que sua soma tenha período comum: frequências incomensuráveis podem produzir um sinal quase-periódico, sem período exato.
- A presença de uma média dependente do tempo viola a estacionariedade fraca; retirar a componente periódica pode deixar um resíduo estacionário, mas isso precisa ser investigado.
- Dados finitos e ruído tornam a identificação visual de periodicidades incerta.
- Um sinal pode ser descrito por harmônicos sem ser estritamente periódico nos instantes observados. A análise de Fourier não requer assumir que qualquer realização finita se repita para sempre.

### 9. Exercícios para sua implementação

1. **Manual:** calcule $\cos(2\pi n/6)$ para $n=0,1,2,3$; encontre um período discreto.
2. **Raciocínio:** a soma de um cosseno de período 6 e outro de período 8 tem qual período comum mínimo? Justifique pelo mínimo múltiplo comum, tomando cuidado com possíveis cancelamentos.
3. **Pseudocódigo:** descreva os passos para criar duas senoides, somá-las e adicionar ruído, antes de escrever o código.
4. **Python:** varie o desvio-padrão do ruído entre 0,2 e 2 e explique o que mudou e o que permaneceu no mecanismo gerador.
5. **Crítico:** forneça um exemplo de sinal que oscila mas não é periódico e explique a diferença entre uma série observada e um processo estocástico.

**Para avançar:** a próxima seção explica unidades de frequência, amplitude, fase e a relação precisa entre seno/cosseno e o modelo linear usado por Morettin e Toloi.
